In [20]:
from references import zeng24_config, zeng24_question
from src.retrieval import VectorRetriever, RerankerManager

In [21]:
cfg = zeng24_config.Zeng24fiqa()

In [22]:
cfg.llm

vLLMConfig(provider='hf', model_name='./Models/Qwen2.5-14B-Instruct', reasoning=True, vllm_parallel_size=2, vllm_gpu_memory_utilization=0.9, temperature=0, top_p=1, max_seq_len=4096, max_gen_len=4096)

In [23]:
# 初始化
retriever = VectorRetriever(cfg, device='cpu', force_rebuild=False)

[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Loading existing Chroma DB: ./data/fiqa
Retriever of similarity_score_threshold is ready.
Retriever of vector-chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!


In [24]:
# 输入查询
queries = [
    "What is stock",
    "Tell me about APPLE company",
]

# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: What is stock
  1. [173088] "What is a stock? A share of stock represents ownership of a portion of a corporation. In olden time...
  2. [330534] For all practical purposes the words mean the same thing.  Shares are just stock in a particular com...
  3. [460058] Stock is a part ownership of a business. First there has to be a business that people want to own pa...
  4. [579919] A share of stock is an asset not much different than any other asset.  If the share is being held in...
  5. [278538] Facebook the company is probably better understood as a capital good - it is a collection of softwar...
  6. [286296] A stock represents your share of ownership in a corporation. All of these shares indicate towards yo...
  7. [509436] Stock is ownership. And whether the thing you own is a good or service irrelevant. The ownership its...
  8. [53993] "A company whose stock is available for sale to the public is called a publicly-held or publicly-tra...
  9. [471964] A share is just a p

In [25]:
reranker = RerankerManager(cfg, device='cpu')

[INFO] Reranker BAAI/bge-reranker-large is ready!


In [26]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



🔍 Query: What is stock
  1. [460058] Stock is a part ownership of a business. First there has to be a business that people want to own pa...
  2. [173088] "What is a stock? A share of stock represents ownership of a portion of a corporation. In olden time...
  3. [286296] A stock represents your share of ownership in a corporation. All of these shares indicate towards yo...

🔍 Query: Tell me about APPLE company
  1. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  2. [135577] They are the largest non government employer in the country. They have an absolutely massive consume...
  3. [527953] "Excellent sagacious analysis, Whale. I agree with every single thing you have stated; however, I to...


In [27]:
from src.llm import OpenAILLM
from src.prompts import SimplePromptConstructor

In [28]:
p_construct = SimplePromptConstructor(cfg)

In [29]:
p_construct.prefix

['context: ', 'question: ', 'answer:']

In [30]:
end_ppt = p_construct.batch_construct(queries, contexts)

In [31]:
print(end_ppt[1])

context: Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  It's also talented at convincing it's fanbase that anything they release is required to live.  Apple fanboys would buy a car just to get the Apple logo somewhere on it.

They are the largest non government employer in the country. They have an absolutely massive consumer base. They made that profit while constantly lowering prices for their customers. I bet you use them on a regular basis. It is kind of funny watching your faux indignation. Isn't Apple your product provider of choice? One of the most expensive over priced producers currently on the market. With one of the lowest paid work forces on the planet. But well paid for their market. You just ooze hypocrite. Like most Apple  social warriors. Its funny listening to people like you speak for others you would never be caught mingling with.

"Excellent sagacious analysis, Whale. I agree with every single thing you have state

In [32]:
LL_Model = OpenAILLM(cfg)

In [33]:
LL_Model.infer("who are you?")

"I'm Qwen, a large language model created by Alibaba Cloud. I'm here to help answer your questions, provide information, and have conversations on a wide range of topics. How can I assist you today?"

In [34]:
LL_Model.batch_infer(end_ppt)

["Stock, also known as equity or share, represents a unit of ownership in a corporation. When you purchase stock in a company, you become a shareholder, meaning you own a piece of that company. The value of your stock can increase or decrease based on the performance and reputation of the company, as well as broader economic conditions and market trends. \n\nStocks are typically traded on stock exchanges, where buyers and sellers come together to exchange shares. These exchanges can be local or international, such as the New York Stock Exchange (NYSE) or NASDAQ in the United States, or the Shanghai Stock Exchange in China. \n\nThe primary reasons why people buy stocks include earning dividends (periodic payments made by corporations to their shareholders), expecting the stock price to rise over time, and participating in the growth and success of the company. However, owning stocks also carries risks, as the value of your investment can fluctuate and potentially decrease.\n\nTo acquire

In [19]:
LL_Model.batch_infer(queries)

['Stock, in the context of finance and business, typically refers to shares of ownership in a corporation or financial asset. When you buy a stock, you\'re essentially buying a small piece of that company. Stocks are also known as "equities" or "shares."\n\nHere are some key points about stocks:\n\n1. **Ownership**: Owning a stock means you own a portion of the company, no matter how small. This gives you certain rights, such as voting on major corporate decisions and receiving dividends (a portion of the company\'s profits).\n\n2. **Market Value**: The value of a stock can fluctuate based on supply and demand, company performance, economic conditions, and other factors.\n\n3. **Types of Stocks**: There are two main types of stocks:\n   - **Common Stock**: Provides voting rights and potential for capital appreciation.\n   - **Preferred Stock**: Often offers higher claim on assets and earnings than common stock but usually doesn\'t come with voting rights.\n\n4. **Trading**: Stocks are 